# GroupBy and Aggregation Lab

In this lab, you'll learn how to use pandas groupby and aggregation operations from first principles. These are essential tools for data analysis that allow you to:

- Group data by categorical variables
- Calculate summary statistics for each group
- Apply custom functions to groups
- Transform and filter grouped data

Let's start by importing pandas and creating a sample dataset that we'll use throughout this lab.

In [1]:
import pandas as pd
import numpy as np

# Create a sample sales dataset
np.random.seed(42)
data = {
    'Region': ['North', 'South', 'East', 'West', 'North', 'South', 'East', 'West'] * 12,
    'Product': ['A', 'B', 'A', 'B', 'C', 'A', 'B', 'C'] * 12,
    'Quarter': ['Q1', 'Q1', 'Q1', 'Q1', 'Q2', 'Q2', 'Q2', 'Q2'] * 12,
    'Sales': np.random.randint(100, 1000, 96),
    'Units': np.random.randint(10, 100, 96),
    'Cost': np.random.randint(50, 500, 96)
}

df = pd.DataFrame(data)


**About the data:** each row is one sale. `Sales` is revenue, `Units` is how many were sold, and `Cost` is what it cost us to produce them.

In [2]:
## YOUR CODE HERE
# Look at the data before you group it:
# 1. Print the first ten rows (head)
print(df.head())
print(df.shape)
print(df.info())
# 2. Print the shape
# 3. Print info on the dataframe (info method)

  Region Product Quarter  Sales  Units  Cost
0  North       A      Q1    202     71   177
1  South       B      Q1    535     23   397
2   East       A      Q1    960     57   280
3   West       B      Q1    370     24   239
4  North       C      Q2    206     81   274
(96, 6)
<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Region   96 non-null     str  
 1   Product  96 non-null     str  
 2   Quarter  96 non-null     str  
 3   Sales    96 non-null     int64
 4   Units    96 non-null     int64
 5   Cost     96 non-null     int64
dtypes: int64(3), str(3)
memory usage: 4.6 KB
None


## Section 1: Basic GroupBy Operations

### What is GroupBy?

The `groupby()` method is the foundation of grouped operations in pandas. It follows the "split-apply-combine" pattern:

1. **Split**: Divide data into groups based on some criteria
2. **Apply**: Apply a function to each group independently
3. **Combine**: Combine the results into a data structure

### Why do we need GroupBy?

Imagine you want to answer: "What are the total sales for each region?" Without groupby, you'd need to:
- Filter for North, sum sales
- Filter for South, sum sales
- Filter for East, sum sales
- Filter for West, sum sales
- Combine results

With groupby, this becomes a single line of code!

### Basic Syntax:
```python
df.groupby('column_name')['value_column'].aggregation_function()
```

Common aggregation functions:
- `.sum()` - Sum of values
- `.mean()` - Average of values
- `.count()` - Count of non-null values
- `.min()` / `.max()` - Minimum/Maximum values
- `.std()` - Standard deviation
- `.median()` - Median value

### Problem 1: Calculate Total Sales by Region

Now it's your turn! Group the data by 'Region' and calculate the total sales for each region.

Store the result in a variable called `sales_by_region`. The result should be a Series with Region as the index.

**Quick check:** once your tests pass, swap `.sum()` for `.mean()` and see how the output changes.

In [8]:
sales_by_region=df.groupby('Region')['Sales'].sum()


print(sales_by_region)

Region
East     14066
North    13971
South    12000
West     11187
Name: Sales, dtype: int64


In [6]:
# Test Problem 1
assert isinstance(sales_by_region, pd.Series), "Result should be a Series"
assert sales_by_region.name == 'Sales', "Series name should be 'Sales'"
assert len(sales_by_region) == 4, "Should have 4 regions"
assert all(region in sales_by_region.index for region in ['North', 'South', 'East', 'West']), "Should have all regions"
assert sales_by_region.sum() == df['Sales'].sum(), "Sum should match total sales"
print("✓ Problem 1 tests passed!")

✓ Problem 1 tests passed!


## Section 2: Multiple Aggregations with `.agg()`

### Why Multiple Aggregations?

Often, a single statistic isn't enough. You might want to see the sum, average, and count all at once to get a complete picture of your data.

### The `.agg()` method

The `.agg()` method (short for aggregate) allows you to apply multiple aggregation functions at once. It's more flexible than calling individual methods.

**Syntax Option 1: List of functions**
```python
df.groupby('column')['value'].agg(['sum', 'mean', 'count', 'std'])
```

**Syntax Option 2: Dictionary for different columns**
```python
df.groupby('column').agg({
    'column1': 'sum',
    'column2': 'mean',
    'column3': ['min', 'max']
})
```

### Problem 2: Multiple Statistics by Region

Group by 'Region' and calculate the following for the 'Sales' column:
- Sum
- Mean
- Count
- Min
- Max

Use the `.agg()` method with a list of aggregation functions.

Store the result in `region_stats`.

In [14]:
region_stats=df.groupby('Region').agg(sum=('Sales','sum'),mean=('Sales','mean'),count=('Sales','count'),min=('Sales','min'),max=('Sales','max'))

print(region_stats)
region_stats.columns

          sum        mean  count  min  max
Region                                    
East    14066  586.083333     24  187  960
North   13971  582.125000     24  113  975
South   12000  500.000000     24  120  997
West    11187  466.125000     24  101  931


Index(['sum', 'mean', 'count', 'min', 'max'], dtype='str')

In [15]:
# Test Problem 2
assert isinstance(region_stats, pd.DataFrame), "Result should be a DataFrame"
assert region_stats.shape == (4, 5), "Should have 4 rows and 5 columns"
assert list(region_stats.columns) == ['sum', 'mean', 'count', 'min', 'max'], "Should have correct column names"
assert region_stats.loc['North', 'sum'] == df[df['Region'] == 'North']['Sales'].sum(), "Sum should be correct"
assert region_stats.loc['North', 'count'] == df[df['Region'] == 'North']['Sales'].count(), "Count should be correct"
print("✓ Problem 2 tests passed!")

✓ Problem 2 tests passed!


### Problem 3: Different Aggregations for Different Columns

Group by 'Product' and calculate:
- Total 'Sales' (sum)
- Average 'Units' (mean)
- Total 'Cost' (sum)

Use `.agg()` with a dictionary to specify different aggregations for different columns.

Store the result in `product_summary`.

Hint: Provide a dictionary with key/value pairs equal to column_name/aggregation_method

In [16]:
product_summary=df.groupby('Product').agg(Sales=('Sales','sum'),Units=('Units','mean'),Cost=('Cost','sum'))

print(product_summary)

         Sales      Units   Cost
Product                         
A        20337  47.972222  10463
B        18237  56.194444  10069
C        12650  62.916667   7341


In [17]:
# Test Problem 3
assert isinstance(product_summary, pd.DataFrame), "Result should be a DataFrame"
assert product_summary.shape == (3, 3), "Should have 3 rows and 3 columns"
assert set(product_summary.columns) == {'Sales', 'Units', 'Cost'}, "Should have correct columns"
assert product_summary.loc['A', 'Sales'] == df[df['Product'] == 'A']['Sales'].sum(), "Sales sum should be correct"
assert abs(product_summary.loc['A', 'Units'] - df[df['Product'] == 'A']['Units'].mean()) < 0.01, "Units mean should be correct"
print("✓ Problem 3 tests passed!")

✓ Problem 3 tests passed!


## Section 3: Grouping by Multiple Columns

### Why Group by Multiple Columns?

Sometimes you need to break down data by multiple dimensions. For example:
- Sales by Region AND Product
- Performance by Department AND Quarter
- Metrics by Customer Segment AND Time Period

### Multi-column GroupBy

You can group by multiple columns by passing a list to `groupby()`:
```python
df.groupby(['column1', 'column2'])
```

This creates a **MultiIndex** (hierarchical index) in the result, which represents all combinations of the grouping columns.


### Problem 4: Sales by Region and Product

Group by both 'Region' and 'Product', then calculate the total 'Sales' for each combination.

Store the result in `sales_by_region_product`.

In [19]:
sales_by_region_product=df.groupby(['Region','Product'])['Sales'].sum()

print(sales_by_region_product)

Region  Product
East    A          8096
        B          5970
North   A          6611
        C          7360
South   A          5630
        B          6370
West    B          5897
        C          5290
Name: Sales, dtype: int64


In [20]:
# Test Problem 4
assert isinstance(sales_by_region_product, pd.Series), "Result should be a Series"
assert isinstance(sales_by_region_product.index, pd.MultiIndex), "Index should be MultiIndex"
assert sales_by_region_product.index.nlevels == 2, "Should have 2 levels in index"
assert len(sales_by_region_product) == 8, "Should have 8 region-product combinations"
assert sales_by_region_product.sum() == df['Sales'].sum(), "Sum should match total sales"
print("✓ Problem 4 tests passed!")

✓ Problem 4 tests passed!


## Section 4: Named Aggregations and Reset Index

### Why Named Aggregations?

When you use `.agg()` with a dictionary, the column names might be unclear or you might want more descriptive names. Named aggregations solve this problem!

### Named Aggregation Syntax

Introduced in pandas 0.25.0, this syntax creates clear, readable column names:
```python
df.groupby('column').agg(
    descriptive_name=('column_to_aggregate', 'function'),
    another_name=('another_column', 'function')
)
```

### The `.reset_index()` Method

After grouping, the grouping columns become the index. Often you want them as regular columns instead:
```python
df.groupby('column')['value'].sum().reset_index()
```

This converts the index back to regular columns and creates a new numeric index.

### Example:

In [21]:
# Example: Named aggregations
example_named = df.groupby('Product').agg(
    total_revenue=('Sales', 'sum'),
    avg_quantity=('Units', 'mean'),
    transaction_count=('Sales', 'count')
)
print("Named Aggregations:")
print(example_named)
print("\n")

# Example: With reset_index
example_reset = df.groupby('Region')['Sales'].sum().reset_index()
print("After reset_index():")
print(example_reset)
print("\nColumn names:", example_reset.columns.tolist())

Named Aggregations:
         total_revenue  avg_quantity  transaction_count
Product                                                
A                20337     47.972222                 36
B                18237     56.194444                 36
C                12650     62.916667                 24


After reset_index():
  Region  Sales
0   East  14066
1  North  13971
2  South  12000
3   West  11187

Column names: ['Region', 'Sales']


### Problem 5: Create a Summary Table with Named Columns

Group by 'Product' and create a summary DataFrame with the following columns:
- `total_sales`: sum of Sales
- `avg_units`: mean of Units  
- `min_cost`: minimum Cost
- `max_cost`: maximum Cost

Use the named aggregation syntax and reset the index so 'Product' becomes a regular column.

Store the result in `product_summary_table`.

In [25]:
product_summary_table=df.groupby('Product').agg(total_sales=('Sales','sum'),avg_units=('Units','mean'),min_cost=('Cost','min'),max_cost=('Cost','max')).reset_index()

print(product_summary_table)

  Product  total_sales  avg_units  min_cost  max_cost
0       A        20337  47.972222        51       495
1       B        18237  56.194444       101       489
2       C        12650  62.916667       146       460


In [26]:
# Test Problem 5
assert isinstance(product_summary_table, pd.DataFrame), "Result should be a DataFrame"
assert list(product_summary_table.columns) == ['Product', 'total_sales', 'avg_units', 'min_cost', 'max_cost'], "Should have correct column names"
assert len(product_summary_table) == 3, "Should have 3 products"
assert 'Product' in product_summary_table.columns, "Product should be a column after reset_index"
assert product_summary_table['total_sales'].sum() == df['Sales'].sum(), "Total sales should match"
print("✓ Problem 5 tests passed!")

✓ Problem 5 tests passed!


## Section 5: Custom Aggregation Functions

### Why Custom Functions?

Built-in aggregations (sum, mean, etc.) are powerful, but sometimes you need custom calculations:
- Range (max - min)
- Coefficient of variation
- Percentiles
- Custom business logic

### Using Lambda Functions

Lambda functions are anonymous functions perfect for simple custom aggregations:
```python
df.groupby('column')['value'].agg(lambda x: x.max() - x.min())
```
x inside the lambda represents the Series of values in the specified column for one group.

### Using Named Functions

For complex logic, define a regular function:
```python
def custom_metric(series):
    # Your custom calculation
    return result

df.groupby('column')['value'].agg(custom_metric)
```

### Example:

In [27]:
# Example: Custom function - coefficient of variation
def coeff_variation(series):
    """Calculate coefficient of variation (std/mean)"""
    return series.std() / series.mean()

example_cv = df.groupby('Product')['Units'].agg(coeff_variation)
print("Coefficient of Variation for Units by Product:")
print(example_cv)
print("\n")

# Example: Multiple custom aggregations
example_custom_multi = df.groupby('Region')['Sales'].agg([
    ('range', lambda x: x.max() - x.min()),
    ('cv', lambda x: x.std() / x.mean()),
    'mean'
])
print("Multiple Custom Aggregations:")
print(example_custom_multi)

Coefficient of Variation for Units by Product:
Product
A    0.507887
B    0.523538
C    0.429073
Name: Units, dtype: float64


Multiple Custom Aggregations:
        range        cv        mean
Region                             
East      773  0.428781  586.083333
North     862  0.451857  582.125000
South     877  0.537676  500.000000
West      830  0.561133  466.125000


### Problem 6: Calculate Range by Product

For each Product, calculate the range (max - min) of Units.

Use a lambda function with `.agg()` and store the result in `units_range_by_product`.

In [31]:
units_range_by_product=df.groupby('Product')['Units'].agg(lambda x:x.max()-x.min())

print(units_range_by_product)

Product
A    87
B    89
C    85
Name: Units, dtype: int64


In [32]:
# Test Problem 6
assert isinstance(units_range_by_product, pd.Series), "Result should be a Series"
assert len(units_range_by_product) == 3, "Should have 3 products"
a_data = df[df['Product'] == 'A']['Units']
assert units_range_by_product['A'] == (a_data.max() - a_data.min()), "Range calculation should be correct"
print("✓ Problem 6 tests passed!")

✓ Problem 6 tests passed!
